# Phase 1 Dataset Analysis — Heart Disease (`heart_disease`)

This notebook loads the saved Phase 1 pipeline outputs for the **UCI/Kaggle Heart Disease Dataset (`heart_disease`)** and documents the analyst's manual insights.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import Image, Markdown, display

DATASET = 'heart_disease'
OUT_DIR = Path('..') / 'outputs' / DATASET
CLEAN_CSV = Path('..') / 'data' / 'cleaned' / f'{DATASET}_cleaned.csv'

if OUT_DIR.exists():
    prof = json.loads((OUT_DIR / 'profile.json').read_text(encoding='utf-8'))
    display(pd.DataFrame(prof['columns'])[['name', 'raw_dtype', 'semantic_type', 'pct_missing', 'n_unique', 'semantic_type_reason']])
    display(Markdown((OUT_DIR / 'cleaning_log.md').read_text(encoding='utf-8')))
    display(Markdown((OUT_DIR / 'scaling_report.md').read_text(encoding='utf-8')))
else:
    print(f'Run `ads run --input data/raw/heart_disease/heart.csv --name heart_disease` to populate {OUT_DIR}.')

## Manual Insights (Written by the Analyst)

<!-- MANUAL:START -->
- **Integer-encoded nominal categories (`cp`, `restecg`, `slope`, `thal`):** In raw UCI Heart Disease files, chest pain type (`cp`: 0-3), resting ECG (`restecg`: 0-2), ST segment slope (`slope`: 0-2), and thalassemia (`thal`: 0-3) are stored as small integers. Treating them as continuous or discrete counts would compute meaningless linear correlations; our `heart_disease.yaml` overrides them to `categorical_nominal`.
- **Data collection quirk in `ca` and `thal`:** In the 303-row Cleveland UCI split (and its 1,025-row Kaggle resampled variant), `thal == 0` and `ca == 4` correspond to coded missing values (`?` in the original UCI `.data` file) that were replaced with integer codes when converted to CSV.
- **Zero-inflated ST depression (`oldpeak`):** Over 30% of patients have `oldpeak == 0.0` (no exercise-induced ST depression), producing strong zero-inflation and right skewness (`skew > 1.2`). Rank-based non-parametric tests (Mann-Whitney U) are strongly preferred over two-sample t-tests for `oldpeak`.
- **Extreme cholesterol outlier (`chol = 564 mg/dL`):** A single postmenopausal female patient exhibits `chol = 564 mg/dL` (well above the `3.0xIQR` fence), which is a clinically plausible familial hypercholesterolemia value and is rightly flagged rather than deleted.
- **Target label inversion caveat across UCI vs Kaggle versions:** In the original UCI Cleveland database, `num > 0` indicates heart disease presence, whereas in the popular 303-row Kaggle `heart.csv` export, `target == 1` often represents the higher-`thalach`/asymptomatic group depending on preprocessing. Analysts must always inspect the sign of `thalach <-> target` and `exang <-> target`.
- **Reference findings for Phase 5 evaluation to recover:** (1) `cp (chest pain type) <-> target`, (2) `thalach (max heart rate) <-> target`, (3) `exang (exercise-induced angina) <-> target`, (4) `oldpeak (ST depression) <-> target`, (5) `ca (number of major vessels) <-> target`, and (6) `thal <-> target`.
<!-- MANUAL:END -->